# **Enforcing Database Security Policies**

- <span style = "color:green">**Policy:**</span> written document (a wish).
- <span style = "color:green">**Enforcement:**</span> system says “no,” masks the data, or leaves a record.
- <span style = "color:green">**Policy Enforcement:**</span> actively blocks, restricts, or records actions that violate the organization’s rules about who can see or change what.

A **security policy** says what should happen, and **enforcement** is the database refusing, hiding, scrambling, or recording the things that shouldn’t: so the rule holds even when someone is careless, curious, or malicious.

<table>
  <tr><th>Layer</th><th>Question it answers</th><th>Mechanisms</th></tr>
  <tr><td>Authentication</td><td>Who are you?</td><td>Entra ID, MFA, managed identities, service principals</td></tr>
  <tr><td>Authorization</td><td>What may you do?</td><td>Roles, GRANT/DENY, least privilege</td></tr>
  <tr><td>Column/Row control</td><td>Which data may you see</td><td>Row-level security, column permissions, dynamic data masking</td></tr>
  <tr><td>Encryption</td><td>Is stolen data readable</td><td>TDE (at rest), TLS (in transit), Always Encrypted</td></tr>
  <tr><td>Auditing</td><td>What happened, and who did it?</td><td>Audit logs, Purview, threat detection</td></tr>
  <tr><td>Network/lifecycle</td><td>Where can you connect from, and for how long?</td><td>Firewalls, private endpoints, password and secret rotation, retention rules</td></tr>
</table>

## **Scenario 1: A clinician tries to see another hospital’s patients (row-level security)**

- **Policy:** Clinicians may only view patients at their own facility.

- This is the MedCore design from the healthcare project. If enforcement lived only in the Power BI report, a clinician with direct SQL access could bypass it. With **row-level security in the database**, it holds no matter how the data is queried:

In [ ]:
CREATE FUNCTION Security.fn_facility_filter(@facility_id INT)
RETURNS TABLE WITH SCHEMABINDING AS
RETURN SELECT 1 AS allowed
FROM dbo.clinician_facility_map m
WHERE m.facility_id = @facility_id
  AND m.user_email = USER_NAME();

CREATE SECURITY POLICY Security.FacilityPolicy
ADD FILTER PREDICATE Security.fn_facility_filter(organization_id) ON dbo.fact_encounters
WITH (STATE = ON);

- A Boston clinician runs SELECT * FROM fact_encounters and gets only Boston rows. The database enforces it, with no app code to forget.

## **Scenario 2: A support analyst sees data they don’t need (dynamic data masking + column permissions)**

- Policy: Support staff can look up a customer but must never see full SSNs or card numbers

In [ ]:
ALTER TABLE dbo.patients
ALTER COLUMN ssn ADD MASKED WITH (FUNCTION = 'partial(0,"XXX-XX-",4)');

GRANT SELECT ON dbo.patients TO support_role;   -- sees XXX-XX-6789
GRANT UNMASK  TO compliance_role;               -- sees the full value

- Same table, same query, different result depending on role. For truly high-risk columns, deny access outright:

In [ ]:
DENY SELECT ON dbo.patients(ssn) TO support_role;

- Masking is a convenience layer, not strong protection, since a determined user with query access can sometimes infer masked values. Use real column permissions or encryption for the most sensitive fields.

## **Scenario 3: An intern accidentally drops a production table (least privilege / RBAC)**

- **Policy:** People and apps get only the permissions their job needs.
- Without enforcement, the intern’s account has db_owner, and one wrong command wipes out Sales.Orders. With role-based access control:

In [ ]:
CREATE ROLE analyst_readonly;
GRANT SELECT ON SCHEMA::gold TO analyst_readonly;
ALTER ROLE analyst_readonly ADD MEMBER [intern@contoso.com];

DENY DELETE, DROP... -- (DROP isn't granted at all, so it's simply refused)

- DROP TABLE now fails with a permission error. The same principle drove our Fabric choices: the Prod service principal was read-only on the source, and the Key Vault role was Key Vault Secrets User, not Contributor.

## **Scenario 4: A stolen laptop and a leaked backup (encryption)**

**Policy:** Data must be unreadable if storage media or backups are stolen.

- **Encryption at rest (TDE):** database files and backups are encrypted on disk. Someone who walks off with a backup file can’t restore or read it without the keys.
- **Encryption in transit (TLS):** a sniffer on the coffee-shop Wi-Fi sees gibberish between the app and the database.
- **Always Encrypted:** for the most sensitive columns, encryption keys stay with the application, so even a DBA with full server access sees only ciphertext.

Fabric and OneLake encrypt data at rest by default, and customer-managed keys exist for organizations whose policy requires holding their own keys.

## **Scenario 5: A suspicious 2 a.m. bulk export (auditing and threat detection)**

- **Policy:** All access to sensitive data is logged, and anomalies are flagged.

In [ ]:
CREATE SERVER AUDIT medcore_audit TO URL (...);
CREATE DATABASE AUDIT SPECIFICATION phi_access
FOR SERVER AUDIT medcore_audit
ADD (SELECT ON dbo.patients BY public)
WITH (STATE = ON);

Weeks later, a compliance officer asks, “Who read the records of this patient?” The audit log answers with user, time, and statement. Threat detection can also alert on patterns such as one account reading millions of rows at 2 a.m., or a classic SQL-injection signature. In the MedCore design, Purview audit and sensitivity labels play this role at the Fabric level.

## **Scenario 6: Contractor access that never gets revoked (lifecycle enforcement)**

**Policy:** Access expires when the need does.

A contractor finishes a six-month engagement, but their account lives on for two years. Enforcement looks like:

- Access granted through **Entra security groups**, so removing someone from the group removes database access everywhere (the pattern from the Fabric RBAC setup).
- **Time-bound access** via Privileged Identity Management, plus periodic access reviews.
- **Secret and token rotation:** SAS tokens with expiry dates, service-principal secrets rotated through Key Vault.
- Disabled accounts and removed role memberships are cleaned up automatically.

## **Scenario 7: Someone tries to connect from an unknown network (network enforcement)**

**Policy:** The database accepts connections only from approved networks.

- **Firewall rules** and **private endpoints** make the database unreachable from the public internet.
- This is exactly the **Trusted Workspace Access setup:** the storage firewall denies everyone except one named Fabric workspace through a resource instance rule.

A stolen password is useless if the attacker can’t even reach the server.

## **Scenario 8: A developer wants to test with production data (separation and masking)**

**Policy:** Dev and Test never contain real patient data.

This is why the healthcare project used synthetic data (Synthea) throughout. If real data must reach lower environments, enforcement means masking or anonymizing it in the copy process, using separate credentials per environment, and blocking cross-environment access. A Dev service principal that can’t authenticate to Prod makes the mistake impossible.

## **Defense in depth**

- No single control is enough. A real incident has to get past all of these:
    - <span style = "color:red">**Network firewall  →  Authentication (MFA)  →  Role permissions  →  Row/column rules →  Encryption  →  Audit logging  →  Alerts and review**</span>

## **Policy vs. enforcement, side by side**

<table>
  <tr><th>Written policy</th><th>How the database enforces it</th></tr>
  <tr><td>“Only HR sees salaries”</td><td>Column DENY, or masked for everyone but hr_role</td></tr>
  <tr><td>“Clinicians see only their facility”</td><td>Row-level security predicate</td></tr>
  <tr><td>“No one edits Prod directly”</td><td>Only the deployment identity holds write rights</td></tr>
  <tr><td>“PHI must be encrypted”</td><td>TDE, TLS, Always Encrypted, sensitivity labels</td></tr>
  <tr><td>“Passwords must rotate every 90 days”</td><td>Expiry settings, or no passwords at all (managed identity)</td></tr>
  <tr><td>“Log every access to patient records”</td><td>Database audit specification</td></tr>
  <tr><td>“Contractors lose access at contract end”</td><td>Group-based access with automatic expiry</td></tr>
</table>

## **Common ways enforcement fails**

- Relying on the application layer only. Anyone who reaches the database directly bypasses the rules.
- Shared admin accounts. The audit log says “sa did it,” which identifies no one.
- Over-broad roles like db_owner for convenience, and permissions that accumulate over time.
- Policies never tested. Impersonate a restricted user and confirm the denial, as in the RLS UAT step in the healthcare guide.
- Forgetting that RLS and role memberships don’t travel through deployment pipelines automatically, so Prod ships open or empty.
- Treating masking as encryption.
- Logging enabled but never reviewed.